# XAI avanzado para mejores modelos

Este notebook ejecuta el analisis XAI reproducible para los modelos campeones de Kepler y TESS sin reentrenar ni modificar sus pesos.

Se generan Saliency Maps, SmoothGrad, Integrated Gradients, Occlusion Sensitivity, Grad-CAM 1D, Grad-CAM multiescala, mapas de consenso, MC Dropout, calibracion post-hoc, contrafactuales simples y resumen por tipo de error. La logica principal vive en `xai_mejores_modelos.py`, por lo que las salidas tambien se pueden regenerar desde consola.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "mejores_resultados").exists() and (ROOT / "proyecto" / "mejores_resultados").exists():
    ROOT = ROOT / "proyecto"

sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)


ROOT = /home/cristobal/Documentos/USM/semestre 11/XAI/proyecto/entrega2/github


## Ejecucion

Los parametros por defecto son los usados para la entrega. Para una prueba rapida se pueden reducir `MC_SAMPLES`, `SMOOTH_SAMPLES`, `IG_STEPS` o `MAX_EXAMPLES`.


In [2]:
import xai_mejores_modelos as xai

MC_SAMPLES = 100
SMOOTH_SAMPLES = 32
IG_STEPS = 64
MAX_EXAMPLES = 6
BATCH_SIZE = 512

resultados = xai.run_xai(
    root=ROOT,
    dataset="both",
    mc_samples=MC_SAMPLES,
    max_examples=MAX_EXAMPLES,
    batch_size=BATCH_SIZE,
    smooth_samples=SMOOTH_SAMPLES,
    ig_steps=IG_STEPS,
)


ModuleNotFoundError: No module named 'h5py'

## Ejemplos explicados


In [ ]:
examples_df = resultados["examples"]
examples_df


## MC Dropout y resultados por tipo de error


In [ ]:
mc_df = resultados["mc"]
mc_df.groupby("dataset")[["mc_mean", "mc_std", "mc_mean_calibrated", "mc_std_calibrated"]].describe()


In [ ]:
outcome_df = resultados["outcome"]
outcome_df


## Calibracion post-hoc

La temperatura se ajusta en validacion y se evalua en test. No cambia los pesos del modelo.


In [ ]:
calibration_df = resultados["calibration"]
calibration_df


In [ ]:
reliability_df = resultados["reliability"]
reliability_df.head(20)


## Metricas de importancia

La fraccion central indica cuanta relevancia cae en la ventana alrededor del transito. Grad-CAM clasico se mantiene como diagnostico; los metodos mas defendibles son Occlusion, Integrated Gradients, SmoothGrad y Consenso.


In [ ]:
importance_df = resultados["importance"]
importance_df


In [ ]:
metric_cols = [
    "saliency_central_importance_ratio",
    "smoothgrad_central_importance_ratio",
    "integrated_gradients_central_importance_ratio",
    "occlusion_central_importance_ratio",
    "gradcam_central_importance_ratio",
    "gradcam_multiscale_central_importance_ratio",
    "consensus_central_importance_ratio",
    "top10_overlap_consensus_integrated_gradients",
    "top10_overlap_consensus_occlusion",
]
importance_df.groupby(["dataset", "view"])[metric_cols].mean().round(3)


## Fidelidad por borrado

Si un mapa es fiel, borrar sus puntos mas importantes deberia reducir la confianza hacia la clase explicada.


In [ ]:
faithfulness_df = resultados["faithfulness"]
faithfulness_df.groupby(["dataset", "method", "deleted_fraction"])["confidence_drop"].mean().unstack("deleted_fraction").round(3)


## Contrafactuales

Estas intervenciones modifican la ventana central del transito y miden el cambio de probabilidad/confianza.


In [ ]:
counterfactual_df = resultados["counterfactual"]
counterfactual_df


In [ ]:
counterfactual_df.groupby(["dataset", "counterfactual"])[["probability_delta", "target_confidence_delta", "prediction_changed"]].mean().round(3)


## Focos de atencion por muestra

Este analisis cuenta componentes conectados de alta relevancia en Saliency target-aware para todo el test set. Sirve como exploracion: casos faciles deberian tender a tener pocos focos compactos, mientras que errores podrian mostrar focos mas dispersos o mayor varianza.


In [ ]:
attention_focus_df = resultados["attention_focus"]
attention_focus_df.head()


In [ ]:
attention_focus_confusion_df = resultados["attention_focus_confusion"]
attention_focus_confusion_df.round(3)


In [ ]:
attention_focus_outcome_df = resultados["attention_focus_outcome"]
attention_focus_outcome_df.round(3)


## Figuras individuales


In [ ]:
from IPython.display import Image, display

for rel_path in examples_df["figure_path"].dropna().unique():
    path = ROOT / rel_path
    print(path)
    display(Image(filename=str(path)))


## Figuras resumen


In [ ]:
patterns = [
    "*_mc_dropout_uncertainty.png",
    "*_mc_dropout_scatter.png",
    "*_aggregate_importance_selected_examples.png",
    "*_faithfulness_deletion_curves.png",
    "*_reliability_calibration.png",
    "*_counterfactual_effects.png",
    "*_outcome_uncertainty_summary.png",
    "*_attention_focus_confusion_matrix.png",
    "*_attention_focus_outcome_boxplot.png",
]

for pattern in patterns:
    for path in sorted((ROOT / "resultados_xai" / "figuras").glob(pattern)):
        print(path)
        display(Image(filename=str(path)))
